<a href="https://colab.research.google.com/github/treborskrub/Three_Agent_Process_Engine_v0_4_1/blob/main/three_agent_process_design_engine_v0_5_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# =============================================================================
# THREE-AGENT PROCESS DESIGN ENGINE v0.5
# AUDIT -> RESOLVE -> PRUNE / RESTORE / PROTECT
# =============================================================================
# Purpose:
# - Build on v0.4.1 Safe Path-Level Adaptive Pruning.
# - Make the decision loop explicit:
#
#       MEASURE -> AUDIT -> RESOLVE -> APPLY BOUNDED CHANGE -> MEASURE
#
# - Audit supplies evidence.
# - Resolve supplies judgment.
# - Prune / restore / protect supplies bounded action.
#
# This is a deterministic abstract process-control model.
# It is not a physical, semantic-reasoning, or quantum-hardware simulation.
# =============================================================================

from dataclasses import dataclass
from copy import deepcopy

EPSILON = 1e-9

# -----------------------------------------------------------------------------
# CONTROL PANEL
# -----------------------------------------------------------------------------
SETSTATE_TASK_SUM = 100.0
MAX_CYCLES = 12

BALANCE_FLOOR = 0.90
WATCH_FLOOR = 0.80
CRITICAL_FLOOR = 0.67
EFFICIENCY_CEILING = 1.05

CRITICAL_EXIT_RATIO = 0.80
MAX_CRITICAL_ENTRIES = 3
WASTEFUL_CONFIRMATION_CYCLES = 2

# -----------------------------------------------------------------------------
# PATH MODEL
# -----------------------------------------------------------------------------
@dataclass
class TaskPath:
    name: str
    contribution: float
    cost: float
    reliability: float
    priority: str
    protected: bool = False
    active: bool = True
    status: str = "ACTIVE"
    monitored_cycles: int = 0

    def value_score(self):
        return (self.contribution * self.reliability) / max(self.cost, EPSILON)

# -----------------------------------------------------------------------------
# BASE MEASUREMENTS
# -----------------------------------------------------------------------------
def active_task_sum(paths):
    return sum(path.contribution for path in paths if path.active)

def active_cost(paths):
    return sum(path.cost for path in paths if path.active)

def priority_rank(priority):
    return {
        "CORE": 3,
        "HIGH": 2,
        "NORMAL": 1,
        "LOW": 0,
    }.get(priority, 0)

def classify_state(ratio):
    if ratio < CRITICAL_FLOOR:
        return "CRITICAL"
    if ratio < WATCH_FLOOR:
        return "SHORTFALL_1_3"
    if ratio < BALANCE_FLOOR:
        return "WATCH"
    if ratio >= EFFICIENCY_CEILING:
        return "EFFICIENCY"
    return "BALANCE"

def direction_from_target(previous_sum, current_sum, setstate):
    previous_distance = abs(1.0 - previous_sum / max(setstate, EPSILON))
    current_distance = abs(1.0 - current_sum / max(setstate, EPSILON))

    if current_distance < previous_distance:
        return "CLOSER"
    if current_distance > previous_distance:
        return "FARTHER"
    return "NO_CHANGE"

def efficiency_status(previous_sum, current_sum, previous_cost, setstate):
    previous_distance = abs(1.0 - previous_sum / max(setstate, EPSILON))
    current_distance = abs(1.0 - current_sum / max(setstate, EPSILON))
    improvement = previous_distance - current_distance
    efficiency = improvement / max(previous_cost, EPSILON)

    if efficiency > 0.0001:
        status = "PRODUCTIVE"
    elif efficiency < -0.0001:
        status = "WASTEFUL"
    else:
        status = "STALLED"

    return status, improvement, efficiency

# -----------------------------------------------------------------------------
# AGENT 1: AUDIT
# Collects measurable evidence. It does not choose an action.
# -----------------------------------------------------------------------------
def audit_agent(paths, setstate, previous_sum, previous_cost,
                critical_entries, wasteful_cycles):
    current_sum = active_task_sum(paths)
    current_cost = active_cost(paths)
    ratio = current_sum / max(setstate, EPSILON)

    direction = direction_from_target(previous_sum, current_sum, setstate)
    efficiency, improvement, efficiency_value = efficiency_status(
        previous_sum, current_sum, previous_cost, setstate
    )

    state = classify_state(ratio)

    active_paths = [path for path in paths if path.active]
    inactive_paths = [path for path in paths if not path.active]

    optional_active = [
        path for path in active_paths
        if not path.protected and path.priority != "CORE"
    ]

    optional_active.sort(
        key=lambda path: (
            priority_rank(path.priority),
            path.value_score(),
            path.reliability,
        )
    )

    restore_candidates = [
        path for path in inactive_paths
        if not path.protected and path.priority != "CORE"
    ]

    restore_candidates.sort(
        key=lambda path: (
            -priority_rank(path.priority),
            -path.value_score(),
            -path.reliability,
        )
    )

    prune_candidate = optional_active[0] if optional_active else None
    restore_candidate = restore_candidates[0] if restore_candidates else None

    if prune_candidate:
        projected_sum_after_prune = current_sum - prune_candidate.contribution
        projected_ratio_after_prune = (
            projected_sum_after_prune / max(setstate, EPSILON)
        )
    else:
        projected_sum_after_prune = current_sum
        projected_ratio_after_prune = ratio

    return {
        "current_sum": current_sum,
        "current_cost": current_cost,
        "ratio": ratio,
        "distance": abs(1.0 - ratio),
        "state": state,
        "direction": direction,
        "efficiency": efficiency,
        "improvement": improvement,
        "efficiency_value": efficiency_value,
        "critical_entries": critical_entries,
        "wasteful_cycles": wasteful_cycles,
        "active_count": len(active_paths),
        "inactive_count": len(inactive_paths),
        "prune_candidate": prune_candidate,
        "restore_candidate": restore_candidate,
        "projected_sum_after_prune": projected_sum_after_prune,
        "projected_ratio_after_prune": projected_ratio_after_prune,
    }

# -----------------------------------------------------------------------------
# AGENT 2: RESOLVE
# Converts audit evidence into a decision.
# It does not change any path by itself.
# -----------------------------------------------------------------------------
def resolve_agent(audit, critical_lockout):
    ratio = audit["ratio"]
    state = audit["state"]

    if audit["critical_entries"] >= MAX_CRITICAL_ENTRIES:
        return {
            "decision_state": "RESET_REQUIRED",
            "action": "RESEED_PROTECTED_CORE",
            "reason": "Critical-entry limit reached; only protected paths remain.",
            "critical_lockout": True,
        }

    if critical_lockout and ratio < CRITICAL_EXIT_RATIO:
        return {
            "decision_state": "CRITICAL_LOCKOUT",
            "action": "PROTECT_AND_REDUCE",
            "reason": (
                f"Protected recovery remains active until ratio reaches "
                f"{CRITICAL_EXIT_RATIO:.2f}."
            ),
            "critical_lockout": True,
        }

    if state == "CRITICAL":
        return {
            "decision_state": "CRITICAL",
            "action": "PROTECT_AND_REDUCE",
            "reason": "Task is below the critical floor; protect core capacity.",
            "critical_lockout": True,
        }

    if state == "SHORTFALL_1_3":
        return {
            "decision_state": "SHORTFALL_1_3",
            "action": "AUDIT_FOR_RECOVERY",
            "reason": "Shortfall requires observation before removal of capacity.",
            "critical_lockout": False,
        }

    if state == "WATCH":
        if audit["direction"] == "CLOSER" and audit["restore_candidate"]:
            return {
                "decision_state": "WATCH",
                "action": "RESTORE_ONE_PATH",
                "reason": "Recovery is improving; restore one worthwhile path.",
                "critical_lockout": False,
            }

        return {
            "decision_state": "WATCH",
            "action": "MONITOR_CANDIDATE",
            "reason": "Watch state lacks confirmed recovery.",
            "critical_lockout": False,
        }

    if state == "EFFICIENCY":
        return {
            "decision_state": "EFFICIENCY",
            "action": "SAFE_PRUNE_OR_MONITOR",
            "reason": "Excess activity requires a bounded efficiency correction.",
            "critical_lockout": False,
        }

    if audit["efficiency"] == "WASTEFUL":
        if audit["wasteful_cycles"] >= WASTEFUL_CONFIRMATION_CYCLES:
            return {
                "decision_state": "BALANCE",
                "action": "SAFE_PRUNE_OR_MONITOR",
                "reason": "Repeated wasteful movement supports candidate review.",
                "critical_lockout": False,
            }

        return {
            "decision_state": "BALANCE",
            "action": "HOLD_AND_AUDIT",
            "reason": "One wasteful cycle is insufficient evidence for pruning.",
            "critical_lockout": False,
        }

    return {
        "decision_state": "BALANCE",
        "action": "RETAIN_AND_AUDIT",
        "reason": "System is balanced without confirmed need for intervention.",
        "critical_lockout": False,
    }

# -----------------------------------------------------------------------------
# AGENT 3: BOUNDED ACTION
# Executes one safe action; never changes more than one optional path per cycle.
# -----------------------------------------------------------------------------
def prune_agent(paths, audit, decision, setstate):
    action = decision["action"]
    changed_paths = []
    result = "NO_PATH_CHANGE"

    if action == "SAFE_PRUNE_OR_MONITOR":
        candidate = audit["prune_candidate"]

        if candidate is None:
            result = "NO_OPTIONAL_PATH_AVAILABLE"

        elif candidate.monitored_cycles < 1:
            candidate.monitored_cycles += 1
            candidate.status = "MONITORED"
            changed_paths.append(candidate.name)
            result = "MONITORED_NOT_PRUNED"

        elif audit["projected_ratio_after_prune"] < BALANCE_FLOOR:
            candidate.monitored_cycles += 1
            candidate.status = "PRUNE_BLOCKED_BY_SAFETY_GUARD"
            changed_paths.append(candidate.name)
            result = "PRUNE_BLOCKED_BY_SAFETY_GUARD"

        else:
            candidate.active = False
            candidate.status = "PRUNED"
            candidate.monitored_cycles = 0
            changed_paths.append(candidate.name)
            result = "PRUNED_SAFELY"

    elif action == "RESTORE_ONE_PATH":
        candidate = audit["restore_candidate"]

        if candidate is None:
            result = "NO_PATH_AVAILABLE_TO_RESTORE"
        else:
            candidate.active = True
            candidate.status = "RESTORED"
            candidate.monitored_cycles = 0
            changed_paths.append(candidate.name)
            result = "PATH_RESTORED"

    elif action == "PROTECT_AND_REDUCE":
        for path in paths:
            if path.protected or path.priority == "CORE":
                if not path.active:
                    path.active = True
                    changed_paths.append(path.name)
                path.status = "CORE_PROTECTED"

        candidate = audit["prune_candidate"]
        if candidate and candidate.active:
            candidate.active = False
            candidate.status = "PROTECTIVE_PRUNE"
            candidate.monitored_cycles = 0
            changed_paths.append(candidate.name)

        result = "CORE_PROTECTED_AND_ONE_OPTIONAL_REDUCED"

    elif action == "RESEED_PROTECTED_CORE":
        for path in paths:
            if path.protected or path.priority == "CORE":
                if not path.active:
                    changed_paths.append(path.name)
                path.active = True
                path.status = "SOURCE_RESEEDED"
            else:
                if path.active:
                    changed_paths.append(path.name)
                path.active = False
                path.status = "RESET_PRUNED"
                path.monitored_cycles = 0

        result = "RESEEDED_FROM_PROTECTED_CORE"

    elif action == "MONITOR_CANDIDATE":
        candidate = audit["prune_candidate"]
        if candidate:
            candidate.monitored_cycles += 1
            candidate.status = "MONITORED"
            changed_paths.append(candidate.name)
            result = "CANDIDATE_MONITORED"
        else:
            result = "NO_OPTIONAL_PATH_TO_MONITOR"

    return {
        "result": result,
        "changed_paths": changed_paths,
    }

# -----------------------------------------------------------------------------
# ENVIRONMENT
# Controlled and deterministic path updates.
# Later, a geometry layer can replace this with connected-node effects.
# -----------------------------------------------------------------------------
def update_active_path_contributions(paths, cycle):
    for path in paths:
        if not path.active:
            continue

        if path.protected:
            change = 0.5
        elif path.reliability >= 0.85:
            change = 1.0
        elif path.reliability >= 0.60:
            change = 0.2
        else:
            change = -1.5

        if cycle % 4 == 0 and not path.protected:
            change -= 0.5

        path.contribution = max(0.0, path.contribution + change)

# -----------------------------------------------------------------------------
# DISPLAY
# -----------------------------------------------------------------------------
def candidate_name(candidate):
    return candidate.name if candidate else "None"

def print_paths(paths):
    print(
        f"{'Path':<16}"
        f"{'Contribution':<15}"
        f"{'Cost':<9}"
        f"{'Reliability':<12}"
        f"{'Priority':<10}"
        f"{'Protected':<11}"
        f"{'Active':<8}"
        f"{'Status'}"
    )
    print("-" * 110)

    for path in paths:
        print(
            f"{path.name:<16}"
            f"{path.contribution:<15.2f}"
            f"{path.cost:<9.2f}"
            f"{path.reliability:<12.2f}"
            f"{path.priority:<10}"
            f"{str(path.protected):<11}"
            f"{str(path.active):<8}"
            f"{path.status}"
        )

# -----------------------------------------------------------------------------
# INITIAL PATHS
# -----------------------------------------------------------------------------
INITIAL_PATHS = [
    TaskPath("Core_Signal",   32.0, 2.0, 0.98, "CORE",   True),
    TaskPath("Core_Memory",   24.0, 2.5, 0.92, "CORE",   True),
    TaskPath("Reliable_Path", 18.0, 2.0, 0.88, "HIGH",   False),
    TaskPath("Medium_Path",   14.0, 4.0, 0.62, "NORMAL", False),
    TaskPath("Weak_Path",     10.0, 6.0, 0.32, "LOW",    False),
]

# -----------------------------------------------------------------------------
# MAIN RUN
# -----------------------------------------------------------------------------
paths = deepcopy(INITIAL_PATHS)

previous_sum = active_task_sum(paths)
previous_cost = max(active_cost(paths), 1.0)

critical_lockout = False
critical_entries = 0
wasteful_cycles = 0
history = []

print("\n" + "=" * 110)
print("THREE-AGENT PROCESS DESIGN ENGINE v0.5")
print("MEASURE -> AUDIT -> RESOLVE -> BOUNDED ACTION -> MEASURE")
print("=" * 110)
print(f"SetState: {SETSTATE_TASK_SUM:.2f}")
print(f"Balance safety floor: {BALANCE_FLOOR:.2f}")
print(f"Initial active task sum: {previous_sum:.2f}")
print("\nINITIAL PATH CONFIGURATION")
print_paths(paths)

for cycle in range(1, MAX_CYCLES + 1):
    # 1. AUDIT: measure and collect evidence
    audit = audit_agent(
        paths=paths,
        setstate=SETSTATE_TASK_SUM,
        previous_sum=previous_sum,
        previous_cost=previous_cost,
        critical_entries=critical_entries,
        wasteful_cycles=wasteful_cycles,
    )

    # Update counters after the audit measurement.
    if audit["efficiency"] == "WASTEFUL":
        wasteful_cycles += 1
    else:
        wasteful_cycles = 0

    if audit["state"] == "CRITICAL":
        critical_entries += 1

    # Refresh audit with the current counters for resolution.
    audit["wasteful_cycles"] = wasteful_cycles
    audit["critical_entries"] = critical_entries

    # 2. RESOLVE: choose one bounded next action
    decision = resolve_agent(audit, critical_lockout)
    critical_lockout = decision["critical_lockout"]

    # 3. ACT: apply the decision safely
    action_result = prune_agent(
        paths=paths,
        audit=audit,
        decision=decision,
        setstate=SETSTATE_TASK_SUM,
    )

    # 4. ENVIRONMENT: calculate next-cycle path contributions
    update_active_path_contributions(paths, cycle)
    next_sum = active_task_sum(paths)

    history.append({
        "cycle": cycle,
        "current_sum": audit["current_sum"],
        "ratio": audit["ratio"],
        "direction": audit["direction"],
        "efficiency": audit["efficiency"],
        "state": audit["state"],
        "decision": decision["decision_state"],
        "action": decision["action"],
        "candidate": candidate_name(audit["prune_candidate"]),
        "projected_ratio": audit["projected_ratio_after_prune"],
        "result": action_result["result"],
        "next_sum": next_sum,
    })

    print("\n" + "-" * 110)
    print(f"CYCLE {cycle}")
    print("-" * 110)
    print(
        f"MEASURE  | Sum: {audit['current_sum']:.2f} | "
        f"Ratio: {audit['ratio']:.3f} | "
        f"State: {audit['state']} | "
        f"Direction: {audit['direction']} | "
        f"Efficiency: {audit['efficiency']}"
    )
    print(
        f"AUDIT    | Wasteful cycles: {wasteful_cycles} | "
        f"Critical entries: {critical_entries} | "
        f"Prune candidate: {candidate_name(audit['prune_candidate'])} | "
        f"Projected ratio after candidate prune: "
        f"{audit['projected_ratio_after_prune']:.3f}"
    )
    print(
        f"RESOLVE  | Decision state: {decision['decision_state']} | "
        f"Action: {decision['action']}"
    )
    print(f"REASON   | {decision['reason']}")
    print(f"ACTION   | {action_result['result']}")

    if action_result["changed_paths"]:
        print("CHANGED  | " + ", ".join(action_result["changed_paths"]))
    else:
        print("CHANGED  | None")

    print(f"NEXT     | Task sum after path update: {next_sum:.2f}")

    previous_sum = audit["current_sum"]
    previous_cost = max(audit["current_cost"], 1.0)

# -----------------------------------------------------------------------------
# FINAL REPORT
# -----------------------------------------------------------------------------
final_sum = active_task_sum(paths)
final_ratio = final_sum / SETSTATE_TASK_SUM
final_distance = abs(1.0 - final_ratio)
final_state = classify_state(final_ratio)

print("\n" + "=" * 110)
print("FINAL PATH CONFIGURATION")
print("=" * 110)
print_paths(paths)

print("\n" + "=" * 110)
print("FINAL AUDIT / RESOLVE HISTORY")
print("=" * 110)
print(
    f"{'Cycle':<7}"
    f"{'Current':<11}"
    f"{'Ratio':<9}"
    f"{'State':<17}"
    f"{'Efficiency':<13}"
    f"{'Candidate':<16}"
    f"{'Action':<27}"
    f"{'Result':<33}"
    f"{'Next'}"
)
print("-" * 110)

for row in history:
    print(
        f"{row['cycle']:<7}"
        f"{row['current_sum']:<11.2f}"
        f"{row['ratio']:<9.3f}"
        f"{row['state']:<17}"
        f"{row['efficiency']:<13}"
        f"{row['candidate']:<16}"
        f"{row['action']:<27}"
        f"{row['result']:<33}"
        f"{row['next_sum']:<10.2f}"
    )

print("=" * 110)
print(f"Final active task sum: {final_sum:.2f}")
print(f"Final ratio r: {final_ratio:.3f}")
print(f"Distance from SetState: {final_distance:.3f}")
print(f"Final state: {final_state}")

if BALANCE_FLOOR <= final_ratio < EFFICIENCY_CEILING:
    verdict = "BALANCE RANGE REACHED"
elif WATCH_FLOOR <= final_ratio < BALANCE_FLOOR:
    verdict = "WATCH / PARTIAL RECOVERY"
elif final_ratio < CRITICAL_FLOOR:
    verdict = "CRITICAL SHORTFALL REMAINS"
elif final_ratio < WATCH_FLOOR:
    verdict = "ONE-THIRD SHORTFALL REMAINS"
else:
    verdict = "EFFICIENCY CORRECTION STILL NEEDED"

print(f"FINAL VERDICT: {verdict}")
print("=" * 110)
print("v0.5 baseline: Audit provides evidence; resolve selects action;")
print("bounded action changes at most one optional path per cycle.")
print("=" * 110)


THREE-AGENT PROCESS DESIGN ENGINE v0.5
MEASURE -> AUDIT -> RESOLVE -> BOUNDED ACTION -> MEASURE
SetState: 100.00
Balance safety floor: 0.90
Initial active task sum: 98.00

INITIAL PATH CONFIGURATION
Path            Contribution   Cost     Reliability Priority  Protected  Active  Status
--------------------------------------------------------------------------------------------------------------
Core_Signal     32.00          2.00     0.98        CORE      True       True    ACTIVE
Core_Memory     24.00          2.50     0.92        CORE      True       True    ACTIVE
Reliable_Path   18.00          2.00     0.88        HIGH      False      True    ACTIVE
Medium_Path     14.00          4.00     0.62        NORMAL    False      True    ACTIVE
Weak_Path       10.00          6.00     0.32        LOW       False      True    ACTIVE

--------------------------------------------------------------------------------------------------------------
CYCLE 1
-----------------------------------------